In [1]:
import dask.dataframe as dd
import pandas as pd
# import trenchripper.trenchripper as tr
import trenchripper as tr
from dask.distributed import wait

# addition of active memory manager
import dask
dask.config.set({'distributed.scheduler.active-memory-manager.start': True});
dask.config.set({'distributed.scheduler.worker-ttl': "5m"});
dask.config.set({'distributed.scheduler.allowed-failures': 100});

dask_wd = "/home/lag36/scratch/lag36/dask"

/home/lag36/miniconda3/envs/crispri/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.0' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [31]:
dask_controller = tr.trcluster.dask_controller(
    walltime="2:00:00",
    local=False,
    n_workers=100,#20, # 200 to run growth function
    n_workers_min=20,#200,
    memory="8GB",
    working_directory=dask_wd,
)
dask_controller.startdask()

# dask_controller = tr.trcluster.dask_controller(
#     walltime="2:00:00",
#     local=False,
#     n_workers=50,
#     n_workers_min=50,
#     memory="8GB",
#     working_directory=dask_wd,
# )
# dask_controller.startdask()
dask_controller.displaydashboard()

110m
2:00:00


I need a full experiment (03/26), the old lLAG8, and the new lLAG8.
 - Full experiment to validate timestamps for the growth df
 - Old lLAG8 to validate timestamps before shifting, and to check the spread of the datapoints

# Validate Lineage dfs

In [2]:
headpath = '/home/lag36/scratch/lag36/'

experiment_directories ={'lLAG08_1-R1': '/2024-08-16_lLAG8_Run-1/', 
                         'lLAG08_1-R2': '/2024-08-16_lLAG8_Run-1_Pipeline-Run-2-2025-05-14/',
                        #  'lLAG08_2-R1': '/2025-02-20_lLAG8-MBM-37C-Run-05_3-Fids_Auto-Switch/',
                         'lLAG08_9-R1': '/2025-03-26_lLAG8-MBM-37C-Run-09_3-Fids_Auto-Switch/',}

parquet_prefixes = {'lLAG08_1-R1': '/2024-08-16_lLAG8_', 
                    'lLAG08_1-R2': '/2025-05-14_lLAG8_MBM_37C_Run-01_PipRun-02_',
                    # 'lLAG08_2-R1': '/2025-02-20_lLAG8-MBM-37C-Run-05_3-Fids_Auto-Switch/',
                    'lLAG08_9-R1': '/2025-03-26_lLAG8_MBM_37_Run9_',}

parquet_names = {'cyc':'Lineage_Cell_Cycle','obs': 'Lineage_Observations',
                 'gro':'Lineage_Growth_Observations', 'bar': 'Final_Barcodes_df'}

In [ ]:
parquet_directories = {key_exp: {key_name: headpath + experiment_directories[key_exp] 
                       + 'Growth_Division' + parquet_prefixes[key_exp]
                       + parquet_names[key_name] 
                       for key_name in parquet_names.keys()}
                       for key_exp in experiment_directories.keys()}      

column_names = {'cyc' : ['phenotype trenchid', 'time (s)', 'Final timepoints',
                         'Delta Timepoints', 'Final time (s)', 'Initial time (s)', 
                         'Delta time (s)', 'Mid-Cycle time (s)'],
                'obs' : ['phenotype trenchid', 'Observation time (s)'],
                'gro' : ['phenotype trenchid', 'Measurement time (s)'],
                'bar' : ['phenotype trenchid', 'timepoints', 'time (s)']} 

column_names_no_trenchid = {
    key: [col for col in value if col != 'phenotype trenchid']
    for key, value in column_names.items()
} 

column_names = column_names_no_trenchid

{'lLAG08_1-R1': {'cyc': '/home/lag36/scratch/lag36//2024-08-16_lLAG8_Run-1/Growth_Division/2024-08-16_lLAG8_Lineage_Cell_Cycle',
  'obs': '/home/lag36/scratch/lag36//2024-08-16_lLAG8_Run-1/Growth_Division/2024-08-16_lLAG8_Lineage_Observations',
  'gro': '/home/lag36/scratch/lag36//2024-08-16_lLAG8_Run-1/Growth_Division/2024-08-16_lLAG8_Lineage_Growth_Observations',
  'bar': '/home/lag36/scratch/lag36//2024-08-16_lLAG8_Run-1/Growth_Division/2024-08-16_lLAG8_Final_Barcodes_df'},
 'lLAG08_1-R2': {'cyc': '/home/lag36/scratch/lag36//2024-08-16_lLAG8_Run-1_Pipeline-Run-2-2025-05-14/Growth_Division/2025-05-14_lLAG8_MBM_37C_Run-01_PipRun-02_Lineage_Cell_Cycle',
  'obs': '/home/lag36/scratch/lag36//2024-08-16_lLAG8_Run-1_Pipeline-Run-2-2025-05-14/Growth_Division/2025-05-14_lLAG8_MBM_37C_Run-01_PipRun-02_Lineage_Observations',
  'gro': '/home/lag36/scratch/lag36//2024-08-16_lLAG8_Run-1_Pipeline-Run-2-2025-05-14/Growth_Division/2025-05-14_lLAG8_MBM_37C_Run-01_PipRun-02_Lineage_Growth_Observations

In [9]:
# Load parquet files
parquet_files = {key_exp: 
                 {key_name: dd.read_parquet(parquet_directories[key_exp][key_name],
                                            calculate_divisions=True,
                                            columns = column_names[key_name])
                  for key_name in parquet_names.keys()
                 }
                  for key_exp in experiment_directories.keys()
                }

In [10]:
parquet_mins = {key_exp:
                {key_name: parquet_files[key_exp][key_name].min().compute()
                 for key_name in parquet_names.keys()}
                for key_exp in experiment_directories.keys()}

parquet_maxs = {key_exp:
                {key_name: parquet_files[key_exp][key_name].max().compute()
                 for key_name in parquet_names.keys()}
                for key_exp in experiment_directories.keys()}

KeyboardInterrupt: 

In [ ]:
# Dictionary of dataframes per name: 'cyc', 'obs', 'gro', 'bar'
max_all = \
{key_df_name:
 pd.DataFrame.from_dict(
     {col_name: # Columns of the df
      {key_exp: parquet_maxs[key_exp][key_df_name][col_name] # Rows of the df
       for key_exp in experiment_directories.keys()
      }
      for col_name in column_names[key_df_name]
     }
 )
 for key_df_name in parquet_names.keys()
}

min_all = \
{key_df_name:
 pd.DataFrame.from_dict(
     {col_name: # Columns of the df
      {key_exp: parquet_mins[key_exp][key_df_name][col_name] # Rows of the df
       for key_exp in experiment_directories.keys()
      }
      for col_name in column_names[key_df_name]
     }
 )
 for key_df_name in parquet_names.keys()
}

In [68]:
max_all['cyc']

,phenotype trenchid,time (s),Final timepoints,Delta Timepoints,Final time (s),Initial time (s),Delta time (s),Mid-Cycle time (s)
lLAG08_1-R1,765628.0,35099.451752,60.0,49.0,35999.643185,34799.174804,29389.610077,35399.408995
lLAG08_1-R2,682503.0,42299.451752,72.0,54.0,43199.643185,41999.174804,32398.956272,42599.408995
lLAG08_9-R1,621227.0,42460.940845,72.0,59.0,43359.834245,42161.048452,35395.893613,42760.441349


In [71]:
min_all['cyc']

,phenotype trenchid,time (s),Final timepoints,Delta Timepoints,Final time (s),Initial time (s),Delta time (s),Mid-Cycle time (s)
lLAG08_1-R1,22.0,600.797509,3.0,2.0,1500.746345,300.831157,1164.040082,900.788751
lLAG08_1-R2,1.0,7800.797509,15.0,2.0,8700.746345,7500.831157,1164.040082,8100.788751
lLAG08_9-R1,2356.0,610.883403,3.0,2.0,1510.871756,306.420376,1172.260337,908.646066


In [72]:
max_all['obs']

,phenotype trenchid,Observation time (s)
lLAG08_1-R1,765628.0,36296.870620
lLAG08_1-R2,682503.0,43496.870620
lLAG08_9-R1,621227.0,43659.641382


In [73]:
min_all['obs']

,phenotype trenchid,Observation time (s)
lLAG08_1-R1,22.0,0.864806
lLAG08_1-R2,1.0,7200.864806
lLAG08_9-R1,2356.0,1.957349


In [103]:
2000/60

33.333333333333336

In [74]:
max_all['gro']

,phenotype trenchid,Measurement time (s)
lLAG08_1-R1,765628.0,35699.579374
lLAG08_1-R2,682503.0,38000.503563
lLAG08_9-R1,621227.0,43060.203111


In [86]:
parquet_file = parquet_files['lLAG08_1-R2']['gro']

In [90]:
parquet_file[parquet_file['Measurement time (s)']>37999].compute()

,phenotype trenchid,Measurement time (s)
Global CellID-Growth Rate Measurement Index,,
4159015801470003,678075,37999.026070
4160000401370003,678084,37999.026070
4160002301440003,678103,37999.026070
4160003101420004,678111,37999.026070
4160003501520004,678115,37999.026070
...,...,...
4186010901320004,682427,38000.503563
4186011601410003,682434,38000.503563
4186013701250004,682455,38000.503563


In [104]:
dask_controller.shutdown()

Done.


2025-06-01 17:39:11,836 - distributed.deploy.adaptive_core - INFO - Adaptive stop


In [99]:
min_all['gro']

,phenotype trenchid,Measurement time (s)
lLAG08_1-R1,22.0,600.815515
lLAG08_1-R2,1.0,600.815515
lLAG08_9-R1,2356.0,607.898294


In [91]:
3600*12

43200

In [98]:
# Time in mins from the last timepoint to the 12 hour mark
# (43496 - 43200)/60
(43659 - 43200)/60

7.65

In [81]:
max_all['bar']

,phenotype trenchid,timepoints,time (s)
lLAG08_1-R1,765628.0,60.0,36296.870620
lLAG08_1-R2,682503.0,72.0,43496.870620
lLAG08_9-R1,621227.0,72.0,43659.641382


In [82]:
min_all['bar']

,phenotype trenchid,timepoints,time (s)
lLAG08_1-R1,22.0,0.0,0.864806
lLAG08_1-R2,1.0,12.0,7200.864806
lLAG08_9-R1,2356.0,0.0,1.957349


In [84]:
45200/3600

12.555555555555555

In [83]:
38000+7200

45200

In [80]:
43000-3600

39400

In [ ]:
max_all